# Ecological evaluation — Colab A100

Evaluate the four released [Model Spec Midtraining](https://arxiv.org/abs/2605.02087) conditions on our current eight-scenario battery. The default `EVAL_SOURCE="released_msm"` runs **instruction-only baseline, environmental MSM, cheese AFT, and MSM + cheese AFT**. Each released adapter is loaded separately onto the pinned pretrained `meta-llama/Llama-3.1-8B`, with its own released tokenizer. This notebook performs inference only.

Two suites run for every released condition:
- **A/B and full-option scoring:** eight families × eight cost levels × two orders × two readouts = 256 prompts. Full-option scores use mean log probability per answer token. Positive ecological-minus-human margins favor the ecological option; the full-option margin is a preference index, not a calibrated choice probability.
- **Maximum tolerable deaths:** eight families × all 24 mappings of `0`, `1`, `10`, `100` onto `A`, `B`, `C`, and `D` = 192 prompts. Normalize the four label scores within each mapping, then average each number's probability over mappings.

Select **A100 40 GB or larger** and run cells in order. Add `HF_TOKEN` to Colab Secrets for the account with Llama base access once approved. The metadata/tokenizer preview uses the public adapter repositories and can run while base access is pending. If result publication is enabled, add `GITHUB_TOKEN` with repository Contents read/write permission.

`EVAL_SOURCE="saved_qwen"` retains the former numeric-only workflow for our seven saved Qwen checkpoints, including H4rmony R1. The training notebooks are now in `notebooks/training/`.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib
from importlib.metadata import version

REPO_URL = "https://github.com/shengweiming/value-misalignment.git"
REPO_DIR = Path("/content/value-misalignment")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
loaded_versions = {
    name: getattr(sys.modules[name], "__version__", None)
    for name in ("transformers", "peft", "accelerate", "huggingface_hub")
    if name in sys.modules
}
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab-eval.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
changed = [name for name, old in loaded_versions.items() if old != version(name.replace("_", "-"))]
if changed or "torchao" in sys.modules:
    raise RuntimeError("Dependencies changed in this warm runtime. Restart the session, then run from the top: " + ", ".join(changed))
for name in list(sys.modules):
    if name == "scripts" or name.startswith("scripts."):
        del sys.modules[name]
importlib.invalidate_caches()
REPOSITORY_COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
print("Repository commit:", REPOSITORY_COMMIT)

In [ ]:
from google.colab import drive
from packaging.version import Version
import torch

drive.mount("/content/drive")
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), "Use a current Colab runtime (PyTorch >= 2.6)."
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Select an A100 GPU runtime."
gpu = torch.cuda.get_device_properties(0)
assert gpu.total_memory / 2**30 >= 38, "Select an A100 40 GB or larger."
print(f"GPU: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB); PyTorch {torch.__version__}")

## Configuration

All four released conditions run by default, one fresh model and adapter at a time. Base and adapter weights use BF16, with FP32 token log probabilities, SDPA, and no quantization. The original `base` column in saved comparison files means the **instruction-only adapter**, not the raw pretrained Llama. Baseline scores are shared across the three treatment comparisons.

`FORCE_EVALUATION=False` reuses only complete bundles matching model revisions, tokenizers, prompt hashes, scoring code, precision, batch size, and environment. Every new bundle is completed locally, copied to Drive, flushed, remounted, and hash-verified. Partial completed comparisons can be recovered on rerun.

In [ ]:
from google.colab import userdata
from scripts.ecological_prompt_sft import NUMERIC_COST_COUNTS

EVAL_SOURCE = "released_msm"  # released_msm | saved_qwen
EVAL_BATCH_SIZE = 2
FORCE_EVALUATION = False
PUBLISH_TO_GITHUB = True
GITHUB_REPOSITORY = "shengweiming/value-misalignment"
GITHUB_BRANCH = "main"
NUMERIC_VALUES = NUMERIC_COST_COUNTS
LOCAL_EVAL_ROOT = Path("/content/value-misalignment-evals/released_environment_msm")
DRIVE_EVAL_ROOT = Path("/content/drive/MyDrive/value-misalignment/released_environment_msm")
assert EVAL_SOURCE in ("released_msm", "saved_qwen")
assert EVAL_BATCH_SIZE >= 1

HF_TOKEN = None
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    pass  # Hugging Face can also use an existing runtime login.
except userdata.NotebookAccessError:
    raise RuntimeError("Grant this notebook access to the HF_TOKEN Colab secret.") from None

GITHUB_TOKEN = None
if PUBLISH_TO_GITHUB:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Add a GITHUB_TOKEN Colab secret, or set PUBLISH_TO_GITHUB=False.") from None
    if not GITHUB_TOKEN:
        raise RuntimeError("GITHUB_TOKEN is empty; publication is enabled.")

## Released models and tokenizer audit

The [instruction-only baseline](https://huggingface.co/chloeli/llama-3.1-8b-baseline) is the shared comparison. The three environmental releases are [MSM](https://huggingface.co/chloeli/llama-3.1-8b-pro-environment-spec-msm), [cheese AFT](https://huggingface.co/chloeli/llama-3.1-8b-pro-environment-spec-cheese-aft), and [MSM + cheese AFT](https://huggingface.co/chloeli/llama-3.1-8b-pro-environment-spec-msm-cheese-aft). Their exact revisions and weight hashes are pinned in `scripts/released_environment_eval.py`.

The paper's MSM ablation includes general instruction tuning; the release card describes it simply as “MSM only.” The public release contains no detailed training-stage manifest, so we label it as the **released MSM ablation**, without treating its stage history as independently verified. The primary midtraining contrast is **MSM + AFT minus AFT**. This battery tests transfer to our dilemmas, not the paper's ordinary preference benchmark.

The next cell downloads public metadata and tokenizers only. It verifies identical tokenizer files across conditions, checks every prompt/candidate boundary, rejects truncation, and requires A-D and A/B answers to be single tokens. It uses the authors' custom chat template and our existing neutral system prompt; no environmental constitution is added at evaluation time.

In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display, Image

if EVAL_SOURCE == "released_msm":
    from dataclasses import asdict
    from scripts.released_environment_eval import (
        BASE_MODEL, BASE_REVISION, RELEASES, SOURCE_RUN_NAME,
        prepare_released_tokenizers,
    )
    released_tokenizers, tokenizer_audits = prepare_released_tokenizers(token=HF_TOKEN)
    display(pd.DataFrame([{"condition": key, **asdict(spec)} for key, spec in RELEASES.items()]))
    print("Underlying pretrained base:", BASE_MODEL, BASE_REVISION)
    display(pd.DataFrame([
        {"condition": key, "suite": suite, **audit}
        for key, value in tokenizer_audits.items()
        for suite, audit in value["suites"].items()
    ]))
    print("All four released tokenizers and all 448 prompts passed the audit.")

## Optional saved Qwen checkpoint

This cell is skipped in the default released-model mode. For `saved_qwen`, choose `CHECKPOINT` below. The original training configuration is reconstructed only to find and verify the saved Drive adapter; it does not start training. This mode retains its numeric-only evaluation.

In [ ]:
if EVAL_SOURCE == "saved_qwen":
    from google.colab import userdata
    import json
    from scripts.ecological_prompt_sft import (
        DilemmaSFTConfig,
        NUMERIC_COST_COUNTS,
        find_compatible_complete_run as find_dilemma_complete_run,
        validate_complete_run as validate_dilemma_complete_run,
    )
    from scripts.harmony_eval.cases import DEFAULT_COST_COUNTS
    from scripts.harmony_sft import (
        SFTConfig as HarmonySFTConfig,
        find_compatible_complete_run as find_harmony_complete_run,
        validate_complete_run as validate_harmony_complete_run,
    )

    CHECKPOINT = "ecological_option_10_epochs"  # Used only in saved_qwen mode
    # harmony_r1 | ecological_prompt_only | ecological_option | ecological_option_10_epochs | human_option | clash_prompt_only | clash_action

    CHECKPOINT_SPECS = {
        "harmony_r1": {
            "source_kind": "harmony",
            "output_slug": "harmony_r1_qwen3_8b",
        },
        "ecological_prompt_only": {
            "source_kind": "dilemma",
            "training_arm": "prompt_only",
            "dataset_path": Path("data/ecological_dilemmas/v1/records.jsonl"),
            "pair_name": None,
            "output_slug": "ecological_dilemma_prompt_qwen3_8b",
        },
        "ecological_option": {
            "source_kind": "dilemma",
            "training_arm": "ecological_option",
            "dataset_path": Path("data/ecological_dilemmas/sft/ecological_option/records.jsonl"),
            "pair_name": None,
            "output_slug": "ecological_dilemma_ecological_option_qwen3_8b",
        },
        "ecological_option_10_epochs": {
            "source_kind": "dilemma",
            "training_arm": "ecological_option",
            "dataset_path": Path("data/ecological_dilemmas/sft/ecological_option/records.jsonl"),
            "pair_name": None,
            "output_slug": "ecological_dilemma_ecological_option_qwen3_8b",
            "num_train_epochs": 10,
        },
        "human_option": {
            "source_kind": "dilemma",
            "training_arm": "human_option",
            "dataset_path": Path("data/ecological_dilemmas/sft/human_option/records.jsonl"),
            "pair_name": None,
            "output_slug": "ecological_dilemma_human_option_qwen3_8b",
        },
        "clash_prompt_only": {
            "source_kind": "dilemma",
            "training_arm": "prompt_only",
            "dataset_path": Path("data/control_dilemmas/clash/v1/records.jsonl"),
            "pair_name": "qwen3_8b_clash_prompt_control_sft",
            "output_slug": "clash_prompt_control_qwen3_8b",
        },
        "clash_action": {
            "source_kind": "dilemma",
            "training_arm": "action",
            "dataset_path": Path("data/control_dilemmas/clash/sft/action/records.jsonl"),
            "pair_name": "qwen3_8b_clash_action_sft",
            "output_slug": "clash_action_qwen3_8b",
        },
    }
    assert CHECKPOINT in CHECKPOINT_SPECS
    SPEC = CHECKPOINT_SPECS[CHECKPOINT]
    NUMERIC_VALUES = NUMERIC_COST_COUNTS
    DRIVE_OUTPUT_ROOT = (
        Path("/content/drive/MyDrive/value-misalignment") / SPEC["output_slug"]
    )
    if SPEC["source_kind"] == "harmony":
        CONFIG = HarmonySFTConfig(
            output_root=Path("/content/evaluation-only-no-training"),
            require_google_drive=False,
            base_model="Qwen/Qwen3-8B",
            dataset_id="neovalle/H4rmony",
            max_length=1024,
            num_train_epochs=3,
            learning_rate=1e-4,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=16,
            lora_rank=16,
            lora_alpha=32,
            lora_dropout=0.05,
            eval_batch_size=4,
            seed=42,
            cost_counts=DEFAULT_COST_COUNTS,
        )
    else:
        CONFIG = DilemmaSFTConfig(
            output_root=Path("/content/evaluation-only-no-training"),
            training_arm=SPEC["training_arm"],
            dataset_path=SPEC["dataset_path"],
            pair_name=SPEC["pair_name"],
            base_model="Qwen/Qwen3-8B",
            model_revision="b968826d9c46dd6066d109eabc6255188de91218",
            max_length=1024,
            num_train_epochs=SPEC.get("num_train_epochs", 3),
            learning_rate=1e-4,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=16,
            lora_rank=16,
            lora_alpha=32,
            lora_dropout=0.05,
            eval_batch_size=4,
            seed=42,
            cost_counts=DEFAULT_COST_COUNTS,
        )

    if SPEC["source_kind"] == "harmony":
        artifacts = find_harmony_complete_run(DRIVE_OUTPUT_ROOT, CONFIG)
        validate_source_run = validate_harmony_complete_run
    else:
        artifacts = find_dilemma_complete_run(DRIVE_OUTPUT_ROOT, CONFIG)
        validate_source_run = validate_dilemma_complete_run
    if artifacts is None:
        raise RuntimeError(
            f"No compatible hash-verified {CHECKPOINT} checkpoint was found under {DRIVE_OUTPUT_ROOT}. "
            "This evaluation-only notebook will not retrain it."
        )
    validate_source_run(artifacts)
    run_metadata = json.loads(artifacts.metadata_path.read_text())
    assert run_metadata["config"]["base_model"] == CONFIG.base_model
    assert run_metadata["config"]["num_train_epochs"] == CONFIG.num_train_epochs
    SOURCE_MODEL_REVISION = run_metadata["resolved_revisions"][CONFIG.base_model]
    if SPEC["source_kind"] == "harmony":
        assert run_metadata["evaluation"]["pair_name"] == "qwen3_8b_harmony_r1_sft"
        training_objective = "H4rmony R1 response-only SFT"
    else:
        training_objective = run_metadata["training_objective"]
        assert training_objective

    print("Selected checkpoint:", CHECKPOINT)
    print("Drive root:", DRIVE_OUTPUT_ROOT)
    print("Verified source run:", artifacts.run_dir)
    print("Training objective:", training_objective)
    print("Resolved base revision:", SOURCE_MODEL_REVISION)
    print("Final adapter:", artifacts.final_adapter_dir)
    print("Permutation-balanced numerical candidates:", NUMERIC_VALUES)
    CONFIG

## Review the existing questions

The numerical audit below shows all 24 mappings and one prompt per family. Released-model mode also previews the A/B and full-option questions. These use the existing scenario bodies, option texts, and cost grid unchanged.

In [ ]:
from IPython.display import Markdown, display
import pandas as pd
from transformers import AutoTokenizer
from scripts.ecological_prompt_sft import (
    EXTREME_V2_NUMERIC_TEMPLATES,
    NUMERIC_CHOICE_LABELS,
    NUMERIC_PERMUTATION_COUNT,
    build_numeric_threshold_cases,
)
from scripts.harmony_eval.scoring import format_causal_prompt

numeric_preview_cases = build_numeric_threshold_cases(NUMERIC_VALUES)
assert len(EXTREME_V2_NUMERIC_TEMPLATES) == 8
assert len(numeric_preview_cases) == 8 * NUMERIC_PERMUTATION_COUNT == 192
assert all(len(case["candidates"]) == len(NUMERIC_VALUES) for case in numeric_preview_cases)
for family in {case["template_family"] for case in numeric_preview_cases}:
    family_cases = [case for case in numeric_preview_cases if case["template_family"] == family]
    assert len(family_cases) == NUMERIC_PERMUTATION_COUNT
    assert len({case["option_mapping"] for case in family_cases}) == NUMERIC_PERMUTATION_COUNT
print(f"Verified {len(numeric_preview_cases)} prompts: 8 scenarios x 24 complete permutations.")
mapping_audit = pd.DataFrame([
    {
        "permutation_index": case["permutation_index"],
        **{candidate["text"]: candidate["value"] for candidate in case["candidates"]},
    }
    for case in numeric_preview_cases[:NUMERIC_PERMUTATION_COUNT]
])
display(mapping_audit)
print("Representative first permutation for each scenario:")
for case in numeric_preview_cases[::NUMERIC_PERMUTATION_COUNT]:
    display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))

if EVAL_SOURCE == "released_msm":
    preview_tokenizer = released_tokenizers["baseline"]
else:
    preview_tokenizer = AutoTokenizer.from_pretrained(
        CONFIG.base_model, revision=SOURCE_MODEL_REVISION, use_fast=True,
    )
formatted_preview = format_causal_prompt(
    preview_tokenizer,
    numeric_preview_cases[0]["prompt"],
    enable_thinking=False,
)
prompt_ids = preview_tokenizer.encode(formatted_preview, add_special_tokens=False)
candidate_token_audit = []
for candidate in numeric_preview_cases[0]["candidates"]:
    scored_text = candidate["text"]
    full_ids = preview_tokenizer.encode(
        formatted_preview + scored_text,
        add_special_tokens=False,
    )
    assert full_ids[:len(prompt_ids)] == prompt_ids
    candidate_ids = full_ids[len(prompt_ids):]
    assert candidate_ids
    candidate_token_audit.append({
        "candidate_label": candidate["text"],
        "mapped_value_in_preview": candidate["value"],
        "scored_text": scored_text,
        "candidate_token_count": len(candidate_ids),
        "candidate_token_ids": candidate_ids,
    })
assert {row["candidate_label"] for row in candidate_token_audit} == set(NUMERIC_CHOICE_LABELS)
assert {row["candidate_token_count"] for row in candidate_token_audit} == {1}
display(pd.DataFrame(candidate_token_audit))
print("A-D are each exactly one scored token; label audit passed.")

In [ ]:
if EVAL_SOURCE == "released_msm":
    from scripts.ecological_prompt_sft import build_supervision_matched_readout_cases
    choice_preview_cases = build_supervision_matched_readout_cases(choice_only=True)
    assert len(choice_preview_cases) == 256
    display(pd.DataFrame(choice_preview_cases).groupby(["readout_type", "readout_variant"]).size())
    for case in choice_preview_cases:
        if case["cost_count"] == 1 and case["readout_variant"] in ("ecological_a", "ecological_first"):
            display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))
    print("Exact released chat rendering for the first numerical prompt:")
    print(formatted_preview)

## Run evaluation

This is the first cell that loads model weights. If Llama access is still pending, rerun this cell after approval with `HF_TOKEN` available. A fresh run scores all four models sequentially; no adapters are stacked or merged. The runner prints each completed condition and its peak GPU allocation.

In [ ]:
if EVAL_SOURCE == "released_msm":
    from scripts.released_environment_eval import run_released_environment_eval
    released_results = run_released_environment_eval(
        local_root=LOCAL_EVAL_ROOT, drive_root=DRIVE_EVAL_ROOT,
        tokenizers=released_tokenizers, tokenizer_audits=tokenizer_audits,
        batch_size=EVAL_BATCH_SIZE, token=HF_TOKEN, force_evaluation=FORCE_EVALUATION,
    )
    for treatment, suites in released_results.items():
        for suite, result in suites.items():
            print(treatment, suite, "— verified Drive bundle:", result.output_dir)
else:
    from scripts.ecological_prompt_sft import run_numeric_threshold_workflow
    numeric_workflow = run_numeric_threshold_workflow(
        artifacts, cost_counts=NUMERIC_VALUES, batch_size=EVAL_BATCH_SIZE,
        force_evaluation=FORCE_EVALUATION,
    )
    print("Verified Drive result:", numeric_workflow.evaluation_artifacts.output_dir)

## Four-model comparison

A/B and full-option summaries first average the two orders for each family and cost. The main summary uses the **56 positive-cost cells**; zero-cost results remain in the saved per-cell table and plots. Numerical distributions average probabilities over all 24 mappings before computing P(0), expected threshold, mode, and median. The shared baseline is included once in these tables.

The contrast table includes every treatment minus baseline and **MSM + AFT minus AFT**. A positive choice-margin change means more ecological preference. A positive P(0) change means more weight on tolerating zero deaths. These are exploratory readouts over the existing eight families.

In [ ]:
if EVAL_SOURCE == "released_msm":
    from scripts.released_environment_eval import collect_condition_rows, choice_summary
    from scripts.ecological_prompt_sft import average_numeric_threshold_probabilities
    from scripts.ecological_prompt_sft.numeric_evaluation import summarize_numeric_threshold_rows

    choice_rows = collect_condition_rows(released_results, "choice")
    numeric_rows = collect_condition_rows(released_results, "numeric")
    assert len(choice_rows) == 4 * 256
    assert len(numeric_rows) == 4 * 192 * 4
    choices = pd.DataFrame(choice_summary(choice_rows))
    positive = choices[choices.cost_count > 0]
    choice_summary_table = positive.groupby(["condition", "readout_type"]).agg(
        mean_margin=("ecological_minus_human", "mean"),
        ecological_choices=("ecological_choice", "sum"),
        ties=("tie", "sum"), cells=("cost_count", "size"),
    )
    assert set(choice_summary_table.cells) == {56}
    display(choice_summary_table)
    display(choices.pivot(index=["readout_type", "template_family", "cost_count"], columns="condition", values="ecological_minus_human"))

    # Numeric helpers group by model_role, so use unique condition names here.
    numeric_by_condition = [{**r, "model_role": r["condition"]} for r in numeric_rows]
    numerical = pd.DataFrame(summarize_numeric_threshold_rows(numeric_by_condition)).rename(columns={"model_role": "condition"})
    display(numerical.pivot(index="template_family", columns="condition", values=[
        "probability_threshold_0", "expected_threshold", "mode_threshold", "median_threshold",
    ]))
    probabilities = pd.DataFrame(average_numeric_threshold_probabilities(numeric_by_condition))
    display(probabilities.pivot(index=["template_family", "candidate_value"], columns="model_role", values="candidate_probability"))

    condition_metrics = positive.groupby(["condition", "readout_type"]).ecological_minus_human.mean().unstack()
    numeric_means = numerical.groupby("condition")[["probability_threshold_0", "expected_threshold"]].mean()
    condition_metrics = condition_metrics.join(numeric_means)
    display(condition_metrics)
    contrasts = []
    for left, right in (("msm", "baseline"), ("aft", "baseline"), ("msm_aft", "baseline"), ("msm_aft", "aft")):
        contrasts.append({"contrast": f"{left} minus {right}", **(condition_metrics.loc[left] - condition_metrics.loc[right]).to_dict()})
    display(pd.DataFrame(contrasts).set_index("contrast"))
    for treatment, suites in released_results.items():
        display(Markdown(f"### {RELEASES[treatment].label} versus instruction-only baseline"))
        for suite in ("choice", "numeric"):
            display(Image(filename=str(suites[suite].plot_path)))

In [ ]:
if EVAL_SOURCE == "saved_qwen":
    from IPython.display import Image
    from scripts.ecological_prompt_sft import average_numeric_threshold_probabilities

    numeric_artifacts = numeric_workflow.evaluation_artifacts
    numeric_scores = pd.read_csv(numeric_artifacts.raw_scores_path)
    numeric_summaries = pd.read_csv(numeric_artifacts.thresholds_path)
    assert len(numeric_scores) == 2 * len(numeric_preview_cases) * len(NUMERIC_VALUES)
    assert set(numeric_scores["model_role"]) == {"base", "aligned"}
    averaged_probabilities = pd.DataFrame(average_numeric_threshold_probabilities(
        numeric_scores.to_dict(orient="records")
    ))
    assert len(averaged_probabilities) == 2 * 8 * len(NUMERIC_VALUES)
    probability_table = averaged_probabilities.pivot(
        index=["template_family", "candidate_value"],
        columns="model_role",
        values="candidate_probability",
    ).sort_index()
    summary_table = numeric_summaries.pivot(
        index="template_family",
        columns="model_role",
        values=[
            "mode_threshold",
            "median_threshold",
            "expected_log1p_threshold",
            "entropy_nats",
            "probability_threshold_0",
            "probability_threshold_1",
            "probability_threshold_10",
            "probability_threshold_100",
        ],
    ).sort_index()
    display(probability_table)
    display(summary_table)
    display(Image(filename=str(numeric_artifacts.plot_path)))
    print("Rendered cases:", numeric_artifacts.rendered_cases_path)
    print("Raw per-permutation label scores:", numeric_artifacts.raw_scores_path)
    print("Permutation-averaged distribution summaries:", numeric_artifacts.thresholds_path)
    print("Completion manifest:", numeric_artifacts.complete_marker_path)

## Publish verified results

Only compact evaluation artifacts are committed: prompts, scores, summaries, plots, provenance, and completion hashes. Released-model runs publish six bundles (three comparisons × two suites), with explicit adapter IDs in every score row. Adapter weights and secrets are not published. If publication is disabled, the verified Drive copies remain available.

In [ ]:
from scripts.ecological_prompt_sft import publish_results_to_github

if PUBLISH_TO_GITHUB:
    if EVAL_SOURCE == "released_msm":
        bundles = [(f"{treatment}/{suite}", result, SOURCE_RUN_NAME)
                   for treatment, suites in released_results.items() for suite, result in suites.items()]
    else:
        bundles = [("saved_qwen/numeric", numeric_workflow.evaluation_artifacts, artifacts.run_dir.name)]
    for label, result, source_name in bundles:
        publication = publish_results_to_github(
            result, source_run_name=source_name, github_repository=GITHUB_REPOSITORY,
            branch=GITHUB_BRANCH, github_token=GITHUB_TOKEN, repo_root=REPO_DIR,
        )
        print(label, "— GitHub publication verified:", publication.html_url)
else:
    print("GitHub publication disabled; verified Drive results remain available.")